In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
from collections import Counter

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

import warnings
warnings.filterwarnings("ignore")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer , AutoConfig

In [ ]:
!pip install opendatasets

In [ ]:
import opendatasets as od
od.download("https://www.kaggle.com/datasets/banuprakashv/news-articles-classification-dataset-for-nlp-and-ml/data")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: t
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/banuprakashv/news-articles-classification-dataset-for-nlp-and-ml


100%|██████████| 5.57M/5.57M [00:00<00:00, 1.02GB/s]

In [ ]:
df_business = pd.read_csv('/content/news-articles-classification-dataset-for-nlp-and-ml/business_data.csv')
df_education = pd.read_csv('/content/news-articles-classification-dataset-for-nlp-and-ml/education_data.csv')
df_entertainment = pd.read_csv('/content/news-articles-classification-dataset-for-nlp-and-ml/entertainment_data.csv')
df_sport = pd.read_csv('/content/news-articles-classification-dataset-for-nlp-and-ml/sports_data.csv')
df_tech = pd.read_csv('/content/news-articles-classification-dataset-for-nlp-and-ml/technology_data.csv')

In [ ]:
df = pd.concat([df_business, df_education, df_entertainment, df_sport, df_tech] , ignore_index=True)

In [ ]:
import numpy as np
import spacy
import re

nlp = spacy.load('en_core_web_sm')

def clean_text(text):
    if text is None:
        return ""

    text = re.sub(r'https\S+' , ' ' , text)
    text = re.sub(r'\W' , ' ' , text)
    text = re.sub(r'\s+[a-zA-Z]\s+' , ' ' , text)
    text = re.sub(r'^[a-zA-Z]\s+', ' ', text)
    text = re.sub(r'\s+' , ' ' , text)

    return text

def processing_text(text):
    if text is None:
        return ""

    text = clean_text(text)

    if not text or text.isspace():
        return ""

    doc = nlp(text)
    words = [word.lemma_ for word in doc if not word.is_stop and not word.is_punct]

    return ' '.join(words)

In [ ]:
df['cleaned_headlines'] = df['headlines'].apply(processing_text)
df['cleaned_description'] = df['description'].apply(processing_text)
df['cleaned_content'] = df['content'].apply(processing_text)

In [ ]:
df['combined_text'] = df['cleaned_headlines'] + ' ' + df['cleaned_description'] + ' ' + df['cleaned_content']

In [ ]:
data = df[['combined_text' , 'category']]

In [ ]:
labels = data['category'].unique()
label2id = {label: i for i , label in enumerate(labels)}
id2label = {i:label for i , label in enumerate(labels)}

In [ ]:
data['labels'] = data['category'].map(label2id)

In [ ]:
data.head()

,combined_text,category,labels
0,Nirmala Sitharaman equal Morarji Desai record ...,business,0
1,densify network want 2 city pair Air India E...,business,0
2,Air India group induct aircraft day year Air I...,business,0
3,Red Sea woes exporter seek increase credit fre...,business,0
4,Air India group induct plane 6 day 2024 kick l...,business,0


In [ ]:
data = data.sample(frac=1).reset_index(drop=True)

In [ ]:
data.head()

,combined_text,category,labels
0,Ajay Devgn reveal dad Veeru run away 13 gangst...,entertainment,2
1,Rishi Kapoor strict boyfriend possessive hus...,entertainment,2
2,west bengal NEET UG 2023 Counselling Registrat...,education,1
3,Windows 11 23h2 update slow app game fix Windo...,technology,4
4,KL Rahul scared get restrict Experts slam In...,sports,3


In [ ]:
train_df, test_df = train_test_split(data[['combined_text' , 'labels']], test_size=0.2, random_state=42)

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
train_dataset = Dataset.from_pandas(train_df[['combined_text', 'labels']])
test_dataset = Dataset.from_pandas(test_df[['combined_text', 'labels']])

In [ ]:
def tokenize_function(example):
    return tokenizer(example["combined_text"], padding="max_length", truncation=True ,  max_length = 128)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
print(train_dataset.column_names)
print(test_dataset.column_names)

['combined_text', 'labels', '__index_level_0__', 'input_ids', 'attention_mask']
['combined_text', 'labels', '__index_level_0__', 'input_ids', 'attention_mask']


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoConfig, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
import numpy as np


# حمل الـ model مباشرة
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=5,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')
    return {'accuracy': acc, 'f1': f1}

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=500,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print("🚀 Starting training...")
trainer.train()
print("✅ Done!")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🚀 Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.072177,0.983500,0.983527
2,0.220100,0.060326,0.984000,0.984026


✅ Done!


In [ ]:
from transformers import pipeline
import os
os.environ["WANDB_DISABLED"] = "true"

pipe = pipeline("text-classification" , model = trainer.model , tokenizer = tokenizer)


In [ ]:
test_news = [
    "Messi scores amazing goal in World Cup final",
    "New AI breakthrough in quantum computing",
    "President announces new policies for education reform",
    "Stock market crashes due to global economic crisis",
    "New blockbuster movie breaks all records"
]

results = pipe(test_news)

for text , result in zip(test_news , results):
    print(f'{text}')
    print(f"Sentiment: {result['label']} , (Score: {result['score']})")
    print()

Messi scores amazing goal in World Cup final
Sentiment: sports , (Score: 0.990882396697998)

New AI breakthrough in quantum computing
Sentiment: technology , (Score: 0.9813466668128967)

President announces new policies for education reform
Sentiment: education , (Score: 0.6404661536216736)

Stock market crashes due to global economic crisis
Sentiment: business , (Score: 0.960933268070221)

New blockbuster movie breaks all records
Sentiment: entertainment , (Score: 0.9507707953453064)



In [ ]:
output_save_dir = "multi-text-classification"
trainer.save_model(output_save_dir)
tokenizer.save_pretrained(output_save_dir)

('multi-text-classification/tokenizer_config.json',
 'multi-text-classification/special_tokens_map.json',
 'multi-text-classification/vocab.txt',
 'multi-text-classification/added_tokens.json',
 'multi-text-classification/tokenizer.json')

In [ ]:
!zip -r multi-text-classification.zip multi-text-classification

  adding: multi-text-classification/ (stored 0%)
  adding: multi-text-classification/training_args.bin (deflated 54%)
  adding: multi-text-classification/special_tokens_map.json (deflated 42%)
  adding: multi-text-classification/tokenizer_config.json (deflated 75%)
  adding: multi-text-classification/vocab.txt (deflated 53%)
  adding: multi-text-classification/model.safetensors (deflated 8%)
  adding: multi-text-classification/config.json (deflated 50%)
  adding: multi-text-classification/tokenizer.json (deflated 71%)
